# 🚁 AeroScan — Pipeline Completo no Google Colab
**Projeto FETIN 2026 | Equipe 49 | Santa Rita do Sapucaí - MG**

Este notebook reproduz **todo o pipeline** do projeto AeroScan:
download dos datasets → unificação → treinamento → métricas → inferência

---
### ⚠️ Antes de começar
1. Vá em **Ambiente de execução → Alterar tipo de ambiente de execução**
2. Selecione **T4 GPU**
3. Clique em **Salvar**
4. Só então execute as células abaixo

---
### 📋 O que este notebook faz
| Etapa | Célula | Tempo estimado |
|-------|--------|----------------|
| Verificar GPU e instalar dependências | 1 | ~2 min |
| Baixar 5 datasets do Roboflow | 2 | ~5 min |
| Unificar datasets e gerar data.yaml | 3 | ~1 min |
| Validar dataset (estatísticas) | 4 | ~30 seg |
| Treinar YOLOv8n por 50 épocas | 5 | ~30-40 min |
| Avaliar métricas e gerar gráficos | 6 | ~2 min |
| Testar inferência em imagem | 7 | ~1 min |
| Baixar modelo e resultados | 8 | ~1 min |

**Tempo total estimado: ~45 minutos na GPU T4**

## ⚙️ Célula 1 — Verificar GPU e Instalar Dependências

Esta célula verifica se a GPU está ativa e instala as bibliotecas necessárias.

**O que ela faz:**
- Roda `nvidia-smi` para confirmar que a GPU T4 está disponível
- Instala `ultralytics` (pacote que contém o YOLOv8)
- Instala `roboflow` (para baixar os datasets)
- Verifica as versões instaladas

**Como saber se funcionou:**
- Deve aparecer o nome da GPU (ex: `Tesla T4`)
- Deve aparecer `✅ CUDA disponível: True`
- Se aparecer `❌ GPU NÃO detectada`, volte e ative a GPU T4

In [ ]:
# Verifica GPU
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if result.returncode == 0:
    linhas = result.stdout.split('\n')
    for linha in linhas:
        if 'Tesla' in linha or 'T4' in linha or 'A100' in linha or 'V100' in linha:
            print(f'✅ GPU detectada: {linha.strip()}')
    print('✅ GPU está ativa!')
else:
    print('❌ GPU NÃO detectada!')
    print('   Vá em: Ambiente de execução → Alterar tipo → T4 GPU → Salvar')
    print('   Depois reinicie e execute novamente')

In [ ]:
# Instala dependências
print('📦 Instalando dependências...')
!pip install ultralytics roboflow --quiet

# Verifica instalação
import ultralytics
import torch
from roboflow import Roboflow

print(f'✅ Ultralytics: {ultralytics.__version__}')
print(f'✅ PyTorch: {torch.__version__}')
print(f'✅ CUDA disponível: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'✅ GPU: {torch.cuda.get_device_name(0)}')
    print(f'✅ VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## 📦 Célula 2 — Baixar Datasets do Roboflow

Esta célula baixa os 5 datasets que formam o dataset do AeroScan.

**O que ela faz:**
- Conecta na API do Roboflow com a chave do projeto
- Baixa cada dataset no formato YOLOv8 (imagens + labels .txt + data.yaml)
- Salva em `/content/datasets/<nome-do-projeto>/`
- Pula datasets que já foram baixados (evita re-download)

**Datasets baixados:**
| Dataset | Classe | Descrição |
|---------|--------|-----------|
| pool-detection-kmqaa | pool | Piscinas aéreas |
| swimming-pools-detection | swimming_pool | Piscinas aéreas |
| swimming-pool-detection | piscina | Piscinas aéreas |
| tire-x4hgu | tire | Pneus |
| wheeltester | wheel | Pneus |

**Como saber se funcionou:**
- Cada dataset deve mostrar `✅ Baixado com sucesso`
- A pasta `/content/datasets/` deve ter 5 subpastas

In [ ]:
import os
from pathlib import Path
from roboflow import Roboflow

# ⚠️ API Key do Roboflow do projeto AeroScan
ROBOFLOW_API_KEY = 'bhNth8QdfWTgdKvg67gX'
RAW_PATH = Path('/content/datasets')
RAW_PATH.mkdir(exist_ok=True)

rf = Roboflow(api_key=ROBOFLOW_API_KEY)

# Lista de datasets: (workspace, project_id, versão)
DATASETS = [
    ('pool-images',                                        'pool-detection-kmqaa',      1),
    ('swimming-pools',                                     'swimming-pools-detection',  1),
    ('piscina-piloto',                                     'swimming-pool-detection',   1),
    ('king-mongkut-university-technology-of-thonburi',     'tire-x4hgu',                1),
    ('testwheel',                                          'wheeltester',               1),
]

print('🔄 Baixando datasets...\n')
baixados = []
falhas   = []

for workspace, project_id, versao in DATASETS:
    destino = RAW_PATH / project_id
    if destino.exists():
        print(f'  ⏭️  {project_id} — já existe, pulando')
        baixados.append(project_id)
        continue
    try:
        project = rf.workspace(workspace).project(project_id)
        project.version(versao).download('yolov8', location=str(destino))
        print(f'  ✅ {project_id} — baixado com sucesso')
        baixados.append(project_id)
    except Exception as e:
        print(f'  ❌ {project_id} — erro: {e}')
        falhas.append(project_id)

print(f'\n📊 Resultado: {len(baixados)} baixados, {len(falhas)} falhas')
if falhas:
    print(f'   Falhas: {falhas}')

## 🔀 Célula 3 — Unificar Datasets

Esta célula resolve um problema importante: cada dataset usa nomes diferentes para as mesmas classes.

**O problema:**
```
Dataset piscinas-1: classe 0 = "swimming_pool"
Dataset piscinas-2: classe 0 = "pool"
Dataset pneus-1:    classe 0 = "tire"
Dataset pneus-2:    classe 0 = "wheel"
```
Se misturarmos direto, o modelo vai confundir tudo.

**O que esta célula faz:**
- Lê o `data.yaml` de cada dataset para descobrir os nomes originais
- Remapeia todos os nomes para o padrão AeroScan: `0=pool`, `1=tire`
- Reescreve cada arquivo `.txt` de label com os índices corretos
- Copia tudo para `/content/aeroscan_dataset/` com prefixo do dataset de origem
- Gera um único `data.yaml` final com as 2 classes

**Como saber se funcionou:**
- Deve mostrar o total de imagens por split (train/val/test)
- Abra um `.txt` de label — deve ter `0` para piscina ou `1` para pneu na primeira coluna
- O `data.yaml` final deve listar: `names: [pool, tire]`

In [ ]:
import shutil
import yaml
from pathlib import Path

BASE = Path('/content/aeroscan_dataset')
for split in ['train', 'val', 'test']:
    (BASE / split / 'images').mkdir(parents=True, exist_ok=True)
    (BASE / split / 'labels').mkdir(parents=True, exist_ok=True)

# Mapeamento de nomes → índice AeroScan
CLASS_MAP = {
    # Piscinas → 0
    'pool': 0, 'swimming_pool': 0, 'swimming-pool': 0,
    'piscina': 0, 'pool_': 0,
    # Pneus → 1
    'tire': 1, 'tyre': 1, 'wheel': 1,
    'car-tire': 1, 'car_tire': 1, 'pneu': 1,
}

def get_class_names(yaml_path):
    with open(yaml_path) as f:
        d = yaml.safe_load(f)
    names = d.get('names', [])
    if isinstance(names, dict):
        names = [names[i] for i in sorted(names.keys())]
    return [n.lower().replace(' ', '_') for n in names]

def remap_labels(src, dst, class_map, original_names):
    linhas_out = []
    with open(src) as f:
        for linha in f:
            partes = linha.strip().split()
            if not partes:
                continue
            idx_original = int(partes[0])
            if idx_original < len(original_names):
                nome = original_names[idx_original]
                novo_idx = class_map.get(nome)
                if novo_idx is not None:
                    partes[0] = str(novo_idx)
                    linhas_out.append(' '.join(partes))
    with open(dst, 'w') as f:
        f.write('\n'.join(linhas_out))

def copiar_split(src_root, split, original_names, prefixo):
    src_img = Path(src_root) / split / 'images'
    src_lbl = Path(src_root) / split / 'labels'
    if not src_img.exists():
        return 0
    count = 0
    for img in src_img.glob('*'):
        lbl = src_lbl / (img.stem + '.txt')
        dst_img = BASE / split / 'images' / f'{prefixo}_{img.name}'
        dst_lbl = BASE / split / 'labels' / f'{prefixo}_{img.stem}.txt'
        shutil.copy(img, dst_img)
        if lbl.exists():
            remap_labels(lbl, dst_lbl, CLASS_MAP, original_names)
        count += 1
    return count

# Processa cada dataset
print('🔀 Unificando datasets...\n')
total = 0
RAW_PATH = Path('/content/datasets')

for dataset_dir in sorted(RAW_PATH.iterdir()):
    yamls = list(dataset_dir.glob('*.yaml'))
    if not yamls:
        continue
    names = get_class_names(yamls[0])
    print(f'  📂 {dataset_dir.name}')
    print(f'     Classes originais: {names}')
    for split in ['train', 'valid', 'val', 'test']:
        split_dst = 'val' if split == 'valid' else split
        n = copiar_split(dataset_dir, split, names, dataset_dir.name[:8])
        if n > 0:
            print(f'     {split_dst}: {n} imagens copiadas')
            total += n
    print()

# Gera data.yaml final
data_yaml = {
    'path':  '/content/aeroscan_dataset',
    'train': 'train/images',
    'val':   'val/images',
    'test':  'test/images',
    'nc':    2,
    'names': ['pool', 'tire'],
}
yaml_path = BASE / 'data.yaml'
with open(yaml_path, 'w') as f:
    yaml.dump(data_yaml, f, allow_unicode=True, sort_keys=False)

print('=' * 50)
print(f'✅ Dataset unificado em: {BASE}')
print(f'   Total de imagens: {total}')
for split in ['train', 'val', 'test']:
    n = len(list((BASE / split / 'images').glob('*')))
    print(f'   {split}: {n} imagens')
print(f'\n📄 data.yaml:')
print(yaml_path.read_text())

## 🔍 Célula 4 — Validar Dataset

Antes de treinar, é importante verificar se o dataset está correto.

**O que esta célula verifica:**
- Quantas imagens têm label e quantas não têm
- Distribuição de classes (quantas piscinas vs pneus)
- Exemplos de imagens do dataset

**O que esperar:**
- A maioria das imagens deve ter label correspondente
- Idealmente as classes devem estar balanceadas (proporção parecida)
- Se uma classe tiver muito menos imagens, o modelo pode ter dificuldade com ela

In [ ]:
from pathlib import Path
from collections import Counter

BASE = Path('/content/aeroscan_dataset')
CLASSES = ['pool (piscina)', 'tire (pneu)']

print('🔍 Validando dataset...\n')

for split in ['train', 'val', 'test']:
    img_dir = BASE / split / 'images'
    lbl_dir = BASE / split / 'labels'

    if not img_dir.exists():
        continue

    imagens = list(img_dir.glob('*'))
    labels  = list(lbl_dir.glob('*.txt'))
    contador_classes = Counter()

    for lbl in labels:
        with open(lbl) as f:
            for linha in f:
                partes = linha.strip().split()
                if partes:
                    contador_classes[int(partes[0])] += 1

    print(f'  📁 {split.upper()}')
    print(f'     Imagens: {len(imagens)}')
    print(f'     Labels:  {len(labels)}')
    print(f'     Anotações por classe:')
    for idx, nome in enumerate(CLASSES):
        qtd = contador_classes.get(idx, 0)
        barra = '█' * min(30, qtd // 10)
        print(f'       {idx} — {nome}: {qtd:4d}  {barra}')
    print()

print('✅ Validação concluída — dataset pronto para treino!')

## 🏋️ Célula 5 — Treinar YOLOv8n

Esta é a célula principal — onde o modelo aprende a detectar piscinas e pneus.

**O que acontece internamente:**
1. Carrega `yolov8n.pt` — modelo base pré-treinado no COCO (80 classes genéricas)
2. Para cada uma das 50 épocas:
   - Vê todas as imagens de treino em batches de 16
   - Tenta detectar os objetos
   - Compara com o gabarito (labels)
   - Calcula o erro (loss)
   - Ajusta os pesos para errar menos
3. Ao final de cada época, avalia no conjunto de validação
4. Salva `best.pt` sempre que o mAP50 melhora

**Augmentações específicas para drone:**
- `flipud=0.5` — flip vertical (drone vê de cima em qualquer orientação)
- `degrees=45` — rotação até 45° (drone gira durante o voo)
- `scale=0.5` — variação de zoom (altitude muda durante o voo)
- `hsv_*` — variação de cores (luz muda com horário e clima)

**O que acompanhar no terminal:**
- Cada linha mostra: `Época X/50 | loss | mAP50`
- O loss deve cair (de ~1.5 para ~0.4)
- O mAP50 deve subir (de ~0.1 para ~0.9+)

**⏱️ Tempo estimado:** 30-40 minutos na GPU T4

In [ ]:
from ultralytics import YOLO

# Carrega modelo base (transfer learning)
model = YOLO('yolov8n.pt')

print('🏋️ Iniciando treinamento...')
print('   Modelo: YOLOv8n (nano)')
print('   Classes: pool (piscina) + tire (pneu)')
print('   Épocas: 50')
print('   Dataset: /content/aeroscan_dataset')
print('   Acompanhe o mAP50 subindo e o loss caindo...\n')

results = model.train(
    data='/content/aeroscan_dataset/data.yaml',
    epochs=50,
    imgsz=640,
    batch=16,
    name='drone_v1',
    project='/content/runs',
    patience=10,        # para early stopping se não melhorar por 10 épocas
    save=True,
    plots=True,         # gera gráficos automáticos
    verbose=True,
    # Augmentações para imagens aéreas de drone
    flipud=0.5,
    fliplr=0.5,
    degrees=45,
    scale=0.5,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
)

print('\n✅ Treinamento concluído!')
print('   Modelo salvo em: /content/runs/drone_v1/weights/best.pt')

## 📊 Célula 6 — Avaliar Métricas e Ver Gráficos

Esta célula avalia o modelo treinado e mostra os resultados.

**Métricas principais:**
- **mAP@50**: métrica principal — quanto mais alto melhor. Nossa meta era 60%
- **Precision**: quando o modelo diz "é um pneu", em quantos % ele está certo
- **Recall**: de todos os pneus reais na imagem, em quantos % o modelo encontra

**Gráficos gerados:**
- `results.png` — curvas de mAP50 e loss por época
- `confusion_matrix.png` — matriz de confusão por classe
- `val_batch0_pred.jpg` — exemplos de detecção na validação

In [ ]:
from ultralytics import YOLO
from IPython.display import Image, display
from pathlib import Path
import json

# Carrega o melhor modelo
model = YOLO('/content/runs/drone_v1/weights/best.pt')

# Avalia no conjunto de validação
print('🔍 Avaliando modelo...\n')
metrics = model.val(
    data='/content/aeroscan_dataset/data.yaml',
    imgsz=640,
    conf=0.25,
    iou=0.5,
)

# Exibe métricas
print('\n' + '='*55)
print('        📈 MÉTRICAS FINAIS — AeroScan')
print('='*55)
print(f'  mAP@50 (geral):      {metrics.box.map50:.4f}  ({metrics.box.map50*100:.1f}%)')
print(f'  mAP@50-95 (geral):   {metrics.box.map:.4f}  ({metrics.box.map*100:.1f}%)')
print(f'  Precision (geral):   {metrics.box.mp:.4f}  ({metrics.box.mp*100:.1f}%)')
print(f'  Recall (geral):      {metrics.box.mr:.4f}  ({metrics.box.mr*100:.1f}%)')
print('-'*55)
print('  Por classe:')
for i, nome in enumerate(['pool (piscina)', 'tire (pneu)']):
    try:
        ap = metrics.box.ap50[i]
        print(f'    {nome:20s}  AP50 = {ap:.4f} ({ap*100:.1f}%)')
    except:
        print(f'    {nome:20s}  dados insuficientes')
print('='*55)

# Verifica meta
meta = 0.60
resultado = metrics.box.map50
if resultado >= meta:
    diff = (resultado - meta) / meta * 100
    print(f'\n  ✅ META ATINGIDA! mAP50 = {resultado:.4f} (+{diff:.0f}% acima da meta de {meta})')
else:
    print(f'\n  ⚠️  Abaixo da meta. mAP50 = {resultado:.4f} (meta: {meta})')

# Salva métricas em JSON
summary = {
    'projeto': 'AeroScan — FETIN 2026',
    'modelo': 'YOLOv8n',
    'classes': ['pool', 'tire'],
    'metricas': {
        'mAP50':     round(float(metrics.box.map50), 4),
        'mAP50_95':  round(float(metrics.box.map), 4),
        'precision': round(float(metrics.box.mp), 4),
        'recall':    round(float(metrics.box.mr), 4),
    },
    'meta_banca': meta,
    'meta_atingida': bool(resultado >= meta),
}
with open('/content/aeroscan_metricas.json', 'w') as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)
print('\n📄 Métricas salvas em: /content/aeroscan_metricas.json')

# Exibe gráficos gerados pelo YOLO
run_dir = Path('/content/runs/drone_v1')
graficos = [
    ('results.png',           '📈 Evolução de mAP50 e Loss por época'),
    ('confusion_matrix.png',  '🔲 Matriz de Confusão'),
    ('val_batch0_pred.jpg',   '🖼️  Exemplos de detecção na validação'),
]
for filename, titulo in graficos:
    path = run_dir / filename
    if path.exists():
        print(f'\n{titulo}')
        display(Image(str(path), width=700))

## 🧪 Célula 7 — Testar Inferência

Testa o modelo em uma imagem real — exatamente o que acontece quando o drone usa a IA em campo.

**Opção A:** Faça upload de uma foto tirada pelo drone de vocês

**Opção B:** Usa automaticamente uma imagem do conjunto de validação

**O que aparece no resultado:**
- Imagem com bounding boxes coloridas
- Cada caixinha mostra: `classe confiança%`
- Verde = pool (piscina) | Azul = tire (pneu)

In [ ]:
from ultralytics import YOLO
from google.colab import files
from IPython.display import Image, display
from pathlib import Path
import os

model = YOLO('/content/runs/drone_v1/weights/best.pt')

# Tenta upload de imagem do drone
print('📁 Faça upload de uma imagem do drone (ou pressione Cancelar para usar imagem de teste):')
try:
    uploaded = files.upload()
    img_path = list(uploaded.keys())[0]
    print(f'✅ Usando imagem enviada: {img_path}')
except:
    # Usa primeira imagem de validação
    val_imgs = list(Path('/content/aeroscan_dataset/val/images').glob('*'))[:1]
    if val_imgs:
        img_path = str(val_imgs[0])
        print(f'ℹ️  Usando imagem de validação: {Path(img_path).name}')
    else:
        print('❌ Nenhuma imagem encontrada')
        img_path = None

if img_path:
    # Roda inferência
    results = model.predict(
        source=img_path,
        conf=0.25,
        iou=0.5,
        save=True,
        project='/content/runs',
        name='inferencia',
        save_txt=True,
        exist_ok=True,
    )

    # Mostra detecções
    result = results[0]
    classes = ['pool (piscina)', 'tire (pneu)']

    print('\n📊 Detecções encontradas:')
    print('-' * 40)
    if len(result.boxes) == 0:
        print('  Nenhum foco detectado nesta imagem')
        print('  Tente reduzir conf para 0.15 se esperava detectar algo')
    else:
        for box in result.boxes:
            cls  = int(box.cls[0])
            conf = float(box.conf[0])
            print(f'  → {classes[cls]:20s}  confiança: {conf:.1%}')

    # Exibe imagem anotada
    pred_path = f'/content/runs/inferencia/{Path(img_path).name}'
    if os.path.exists(pred_path):
        print('\n🖼️  Imagem com detecções:')
        display(Image(pred_path, width=700))

## 💾 Célula 8 — Baixar Modelo e Resultados

⚠️ **IMPORTANTE:** O Colab reseta quando você fecha ou após inatividade.
Tudo em `/content/` some. **Baixe os arquivos agora!**

**O que será baixado:**
- `best.pt` — o modelo treinado (arquivo mais importante)
- `aeroscan_metricas.json` — métricas para o relatório
- `results.png` — gráfico de treinamento para os slides
- `confusion_matrix.png` — matriz de confusão

**Depois de baixar:**
- Coloque o `best.pt` em `runs/drone_v1/weights/` no seu repositório
- Use o `results.png` no slide de métricas da apresentação

In [ ]:
import zipfile
from google.colab import files
from pathlib import Path

# Arquivos para baixar
arquivos = [
    '/content/runs/drone_v1/weights/best.pt',
    '/content/aeroscan_metricas.json',
    '/content/runs/drone_v1/results.png',
    '/content/runs/drone_v1/confusion_matrix.png',
    '/content/aeroscan_dataset/data.yaml',
]

# Cria ZIP
zip_path = '/content/aeroscan_modelo_final.zip'
with zipfile.ZipFile(zip_path, 'w') as zf:
    for f in arquivos:
        p = Path(f)
        if p.exists():
            zf.write(f, p.name)
            print(f'  ✅ Adicionado: {p.name} ({p.stat().st_size / 1024:.0f} KB)')
        else:
            print(f'  ⚠️  Não encontrado: {p.name}')

zip_size = Path(zip_path).stat().st_size / 1024 / 1024
print(f'\n📦 ZIP criado: {zip_size:.1f} MB')
print('📥 Iniciando download...')
files.download(zip_path)

print('\n' + '='*50)
print('✅ PIPELINE COMPLETO!')
print('   ✔ Datasets baixados e unificados')
print('   ✔ Modelo YOLOv8n treinado')
print('   ✔ Métricas avaliadas e salvas')
print('   ✔ Inferência testada')
print('   ✔ Arquivos baixados')
print('='*50)

## 🔧 Células Extras (Opcional)

As células abaixo são opcionais — úteis para exploração e ajuste fino.

In [ ]:
# OPCIONAL: Exportar para ONNX (para rodar na Raspberry Pi)
# O formato ONNX roda sem precisar do PyTorch instalado

from ultralytics import YOLO
import os

model = YOLO('/content/runs/drone_v1/weights/best.pt')

print('📦 Exportando para ONNX...')
model.export(
    format='onnx',
    imgsz=640,
    simplify=True,
    opset=12,
)

onnx_path = '/content/runs/drone_v1/weights/best.onnx'
if os.path.exists(onnx_path):
    pt_size   = os.path.getsize('/content/runs/drone_v1/weights/best.pt') / 1024 / 1024
    onnx_size = os.path.getsize(onnx_path) / 1024 / 1024
    print(f'\n✅ ONNX exportado!')
    print(f'   best.pt:   {pt_size:.1f} MB (PyTorch — para continuar treinando)')
    print(f'   best.onnx: {onnx_size:.1f} MB (ONNX — para o drone/Raspberry Pi)')

    from google.colab import files
    files.download(onnx_path)

In [ ]:
# OPCIONAL: Retreinar a partir do best.pt (continuar de onde parou)
# Útil se quiser mais épocas sem começar do zero

from ultralytics import YOLO

model = YOLO('/content/runs/drone_v1/weights/best.pt')  # carrega modelo já treinado

results = model.train(
    data='/content/aeroscan_dataset/data.yaml',
    epochs=30,          # épocas ADICIONAIS
    imgsz=640,
    batch=16,
    name='drone_v2',    # salva como v2 para não sobrescrever o v1
    project='/content/runs',
    resume=False,
)
print('✅ Retreino concluído! Modelo salvo em: /content/runs/drone_v2/weights/best.pt')